# Notebook 04: Business Insights and Recommendations

## Project: Retail Sales Intelligence

### Purpose
This notebook translates the findings from the exploratory and commercial performance analysis into practical business insights and recommendations.

The analysis focuses on:
- Overall sales and profitability.
- Product and category performance.
- Store performance.
- Customer purchasing behaviour.
- Commercial risks and opportunities.
- Evidence-based management recommendations.

### Expected outcome
A set of prioritised recommendations supported by quantitative evidence, together with exported tables that can be used in the final project report and Power BI dashboard.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# Project directories
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "eda_outputs"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "business_insights_outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = DATA_DIR / "sales_enriched_eda.csv"

print("Project directory:", PROJECT_ROOT)
print("Input file:", INPUT_FILE)
print("Output directory:", OUTPUT_DIR)

Project directory: C:\Users\Administrator\retail-sales-intelligence
Input file: C:\Users\Administrator\retail-sales-intelligence\data\processed\eda_outputs\sales_enriched_eda.csv
Output directory: C:\Users\Administrator\retail-sales-intelligence\data\processed\business_insights_outputs


In [2]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {INPUT_FILE}. "
        "Check that Notebook 02 ran successfully and exported sales_enriched_eda.csv."
    )

sales = pd.read_csv(INPUT_FILE)

print("Dataset loaded successfully.")
print("Rows:", len(sales))
print("Columns:", len(sales.columns))

display(sales.head())

Dataset loaded successfully.
Rows: 62884
Columns: 35


,order_number,line_item,order_date,delivery_date,customer_key,store_key,product_key,quantity,currency_code,product_name,...,continent,birthday,country_y,state_y,square_meters,open_date,revenue_usd,product_cost_usd,gross_profit_usd,gross_profit_margin
0,366000,1,2016-01-01,NaN,265598,10,1304,1,CAD,Contoso Lens Adapter M450 White,...,North America,1971-03-23,Canada,Nunavut,1210.0,2015-04-04,68.00,31.27,36.73,54.014706
1,366001,1,2016-01-01,2016-01-13,1269051,0,1048,2,USD,A. Datum SLR Camera X136 Silver,...,North America,1995-11-20,Online,Online,NaN,2010-01-01,854.00,282.94,571.06,66.868852
2,366001,2,2016-01-01,2016-01-13,1269051,0,2007,1,USD,Fabrikam Microwave 1.5CuFt X1100 Black,...,North America,1995-11-20,Online,Online,NaN,2010-01-01,665.94,220.64,445.30,66.867886
3,366002,1,2016-01-01,2016-01-12,266019,0,1106,7,CAD,Contoso SLR Camera M146 Orange,...,North America,1962-08-24,Online,Online,NaN,2010-01-01,2254.00,1036.56,1217.44,54.012422
4,366002,2,2016-01-01,2016-01-12,266019,0,373,1,CAD,Adventure Works Laptop8.9 E0890 White,...,North America,1962-08-24,Online,Online,NaN,2010-01-01,326.00,166.20,159.80,49.018405


In [3]:
# Standardise date field
if "order_date" not in sales.columns:
    raise KeyError("The dataset must contain an 'order_date' column.")

sales["order_date"] = pd.to_datetime(sales["order_date"], errors="coerce")

# Convert financial and quantity fields to numeric
numeric_columns = [
    "quantity",
    "unit_price_usd",
    "unit_cost_usd",
    "revenue_usd",
    "product_cost_usd",
    "gross_profit_usd",
    "gross_profit_margin"
]

for column in numeric_columns:
    if column in sales.columns:
        sales[column] = (
            sales[column]
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
        )
        sales[column] = pd.to_numeric(sales[column], errors="coerce")

# Calculate missing financial measures where possible
if "quantity" not in sales.columns:
    raise KeyError("The dataset must contain a 'quantity' column.")

if "revenue_usd" not in sales.columns:
    if "unit_price_usd" not in sales.columns:
        raise KeyError("Cannot calculate revenue: unit_price_usd is missing.")
    sales["revenue_usd"] = sales["quantity"] * sales["unit_price_usd"]

if "product_cost_usd" not in sales.columns:
    if "unit_cost_usd" in sales.columns:
        sales["product_cost_usd"] = sales["quantity"] * sales["unit_cost_usd"]

if "gross_profit_usd" not in sales.columns:
    if "product_cost_usd" in sales.columns:
        sales["gross_profit_usd"] = (
            sales["revenue_usd"] - sales["product_cost_usd"]
        )

if "gross_profit_margin" not in sales.columns:
    if "gross_profit_usd" in sales.columns:
        sales["gross_profit_margin"] = np.where(
            sales["revenue_usd"] != 0,
            sales["gross_profit_usd"] / sales["revenue_usd"] * 100,
            np.nan
        )

# Remove rows without a valid transaction date
sales = sales.dropna(subset=["order_date"]).copy()

# Create useful time fields
sales["year"] = sales["order_date"].dt.year
sales["month"] = sales["order_date"].dt.month
sales["year_month"] = sales["order_date"].dt.to_period("M").astype(str)

print("Prepared dataset:", sales.shape)
print("\nAvailable analytical columns:")
print(sales.columns.tolist())

display(sales.head())

Prepared dataset: (62884, 38)

Available analytical columns:
['order_number', 'line_item', 'order_date', 'delivery_date', 'customer_key', 'store_key', 'product_key', 'quantity', 'currency_code', 'product_name', 'brand', 'color', 'unit_cost_usd', 'unit_price_usd', 'subcategory_key', 'subcategory', 'category_key', 'category', 'gender', 'name', 'city', 'state_code', 'state_x', 'zip_code', 'country_x', 'continent', 'birthday', 'country_y', 'state_y', 'square_meters', 'open_date', 'revenue_usd', 'product_cost_usd', 'gross_profit_usd', 'gross_profit_margin', 'year', 'month', 'year_month']


,order_number,line_item,order_date,delivery_date,customer_key,store_key,product_key,quantity,currency_code,product_name,...,state_y,square_meters,open_date,revenue_usd,product_cost_usd,gross_profit_usd,gross_profit_margin,year,month,year_month
0,366000,1,2016-01-01,NaN,265598,10,1304,1,CAD,Contoso Lens Adapter M450 White,...,Nunavut,1210.0,2015-04-04,68.00,31.27,36.73,54.014706,2016,1,2016-01
1,366001,1,2016-01-01,2016-01-13,1269051,0,1048,2,USD,A. Datum SLR Camera X136 Silver,...,Online,NaN,2010-01-01,854.00,282.94,571.06,66.868852,2016,1,2016-01
2,366001,2,2016-01-01,2016-01-13,1269051,0,2007,1,USD,Fabrikam Microwave 1.5CuFt X1100 Black,...,Online,NaN,2010-01-01,665.94,220.64,445.30,66.867886,2016,1,2016-01
3,366002,1,2016-01-01,2016-01-12,266019,0,1106,7,CAD,Contoso SLR Camera M146 Orange,...,Online,NaN,2010-01-01,2254.00,1036.56,1217.44,54.012422,2016,1,2016-01
4,366002,2,2016-01-01,2016-01-12,266019,0,373,1,CAD,Adventure Works Laptop8.9 E0890 White,...,Online,NaN,2010-01-01,326.00,166.20,159.80,49.018405,2016,1,2016-01


In [4]:
def distinct_count(data, column):
    if column in data.columns:
        return data[column].nunique()
    return np.nan

total_revenue = sales["revenue_usd"].sum()
total_profit = sales["gross_profit_usd"].sum() if "gross_profit_usd" in sales.columns else np.nan
total_units = sales["quantity"].sum()

total_orders = distinct_count(sales, "order_number")
total_customers = distinct_count(sales, "customer_key")
total_products = distinct_count(sales, "product_key")
total_stores = distinct_count(sales, "store_key")

overall_margin = (
    total_profit / total_revenue * 100
    if pd.notna(total_profit) and total_revenue != 0
    else np.nan
)

business_baseline = pd.DataFrame({
    "Metric": [
        "Total Revenue (USD)",
        "Total Gross Profit (USD)",
        "Gross Profit Margin (%)",
        "Units Sold",
        "Number of Orders",
        "Unique Customers",
        "Unique Products",
        "Number of Stores"
    ],
    "Value": [
        total_revenue,
        total_profit,
        overall_margin,
        total_units,
        total_orders,
        total_customers,
        total_products,
        total_stores
    ]
})

display(business_baseline)

,Metric,Value
0,Total Revenue (USD),5.575548e+07
1,Total Gross Profit (USD),3.266269e+07
2,Gross Profit Margin (%),5.858202e+01
3,Units Sold,1.977570e+05
4,Number of Orders,2.632600e+04
5,Unique Customers,1.188700e+04
6,Unique Products,2.492000e+03
7,Number of Stores,5.800000e+01


In [5]:
monthly_sales = (
    sales.groupby("year_month", as_index=False)
    .agg(
        revenue_usd=("revenue_usd", "sum"),
        gross_profit_usd=("gross_profit_usd", "sum"),
        units_sold=("quantity", "sum")
    )
    .sort_values("year_month")
)

monthly_sales["revenue_growth_pct"] = (
    monthly_sales["revenue_usd"].pct_change() * 100
)

monthly_sales["profit_growth_pct"] = (
    monthly_sales["gross_profit_usd"].pct_change() * 100
)

best_month = monthly_sales.loc[monthly_sales["revenue_usd"].idxmax()]
weakest_month = monthly_sales.loc[monthly_sales["revenue_usd"].idxmin()]

print("Highest-revenue month:", best_month["year_month"])
print(f"Revenue: ${best_month['revenue_usd']:,.2f}")

print("\nLowest-revenue month:", weakest_month["year_month"])
print(f"Revenue: ${weakest_month['revenue_usd']:,.2f}")

display(monthly_sales.tail(12))

Highest-revenue month: 2019-12
Revenue: $2,477,295.85

Lowest-revenue month: 2017-04
Revenue: $58,580.07


,year_month,revenue_usd,gross_profit_usd,units_sold,revenue_growth_pct,profit_growth_pct
50,2020-03,674990.66,396204.76,2392,-69.695751,-69.788784
51,2020-04,217642.10,128029.54,694,-67.756280,-67.686017
52,2020-05,878026.63,514047.95,3256,303.426832,301.507301
53,2020-06,763883.23,443405.96,2738,-12.999993,-13.742296
54,2020-07,512805.18,302322.37,1941,-32.868643,-31.818154
55,2020-08,416647.08,245161.15,1526,-18.751390,-18.907374
56,2020-09,380431.52,222284.38,1328,-8.692143,-9.331319
57,2020-10,245647.59,141098.51,1042,-35.429223,-36.523426
58,2020-11,256701.02,149679.25,1044,4.499710,6.081382
59,2020-12,651526.44,384657.66,2411,153.807499,156.987966


In [6]:
category_performance = pd.DataFrame()

if "category" in sales.columns:
    category_performance = (
        sales.groupby("category", dropna=False)
        .agg(
            revenue_usd=("revenue_usd", "sum"),
            gross_profit_usd=("gross_profit_usd", "sum"),
            units_sold=("quantity", "sum"),
            transaction_rows=("quantity", "size")
        )
        .reset_index()
    )

    category_performance["gross_margin_pct"] = np.where(
        category_performance["revenue_usd"] != 0,
        category_performance["gross_profit_usd"]
        / category_performance["revenue_usd"] * 100,
        np.nan
    )

    category_performance["revenue_contribution_pct"] = (
        category_performance["revenue_usd"] / total_revenue * 100
        if total_revenue != 0 else np.nan
    )

    category_performance = category_performance.sort_values(
        "revenue_usd", ascending=False
    )

    display(category_performance)

    print("Top revenue-generating category:")
    display(category_performance.head(1))

else:
    print("Category field is unavailable. Category analysis skipped.")

,category,revenue_usd,gross_profit_usd,units_sold,transaction_rows,gross_margin_pct,revenue_contribution_pct
3,Computers,19301595.46,11277447.90,44151,14025,58.427543,34.618293
5,Home Appliances,10795478.59,6296338.85,18401,5865,58.323851,19.362184
1,Cameras and camcorders,6520168.02,3919800.99,17609,5559,60.118098,11.694219
2,Cell phones,6183791.22,3498626.54,31477,10158,56.577372,11.090912
7,TV and Video,5928982.69,3536694.39,11236,3558,59.650948,10.633901
0,Audio,3169627.74,1827851.77,23490,7489,57.667711,5.684872
6,"Music, Movies and Audio Books",3131006.44,1909259.17,28802,9169,60.979088,5.615603
4,Games and Toys,724829.43,396668.77,22591,7061,54.725809,1.300015


Top revenue-generating category:


,category,revenue_usd,gross_profit_usd,units_sold,transaction_rows,gross_margin_pct,revenue_contribution_pct
3,Computers,19301595.46,11277447.9,44151,14025,58.427543,34.618293


In [7]:
product_performance = pd.DataFrame()

if "product_key" in sales.columns:
    product_group_columns = ["product_key"]

    for column in ["product_name", "brand", "category"]:
        if column in sales.columns:
            product_group_columns.append(column)

    product_performance = (
        sales.groupby(product_group_columns, dropna=False)
        .agg(
            revenue_usd=("revenue_usd", "sum"),
            gross_profit_usd=("gross_profit_usd", "sum"),
            units_sold=("quantity", "sum"),
            transaction_rows=("quantity", "size")
        )
        .reset_index()
    )

    product_performance["gross_margin_pct"] = np.where(
        product_performance["revenue_usd"] != 0,
        product_performance["gross_profit_usd"]
        / product_performance["revenue_usd"] * 100,
        np.nan
    )

    product_performance = product_performance.sort_values(
        "revenue_usd", ascending=False
    )

    print("Top 10 products by revenue")
    display(product_performance.head(10))

    print("10 lowest-revenue products")
    display(product_performance.tail(10))

else:
    print("Product key unavailable. Product analysis skipped.")

Top 10 products by revenue


,product_key,product_name,brand,category,revenue_usd,gross_profit_usd,units_sold,transaction_rows,gross_margin_pct
442,444,WWI Desktop PC2.33 X2330 Black,Wide World Importers,Computers,505450.00,337986.00,550,154,66.868335
414,416,Adventure Works Desktop PC2.33 XD233 Silver,Adventure Works,Computers,466089.00,311663.95,481,146,66.867905
426,428,Adventure Works Desktop PC2.33 XD233 Brown,Adventure Works,Computers,464151.00,310368.05,479,144,66.867905
420,422,Adventure Works Desktop PC2.33 XD233 Black,Adventure Works,Computers,447678.00,299352.90,462,151,66.867905
431,433,Adventure Works Desktop PC2.33 XD233 White,Adventure Works,Computers,437019.00,292225.45,451,142,66.867905
453,455,WWI Desktop PC2.33 X2330 White,Wide World Importers,Computers,424578.00,283908.24,462,131,66.868335
448,450,WWI Desktop PC2.33 X2330 Brown,Wide World Importers,Computers,422740.00,282679.20,460,151,66.868335
146,147,"Adventure Works 52"" LCD HDTV X590 White",Adventure Works,TV and Video,394398.64,263727.12,136,40,66.868162
145,146,"Adventure Works 52"" LCD HDTV X590 Black",Adventure Works,TV and Video,374098.71,250152.93,129,44,66.868162
436,438,WWI Desktop PC2.33 X2330 Silver,Wide World Importers,Computers,360248.00,240891.84,392,142,66.868335


10 lowest-revenue products


,product_key,product_name,brand,category,revenue_usd,gross_profit_usd,units_sold,transaction_rows,gross_margin_pct
922,924,SV USB Sync Charge Cable E700 Black,Southridge Video,Computers,37.81,18.62,19,6,49.246231
921,923,SV USB Sync Charge Cable E700 Blue,Southridge Video,Computers,35.82,17.64,18,9,49.246231
918,920,SV USB Data Cable E600 Black,Southridge Video,Computers,30.40,15.04,32,10,49.473684
2413,2439,Litware 120mm Blue LED Case Fan E901 Black,Litware,Home Appliances,29.97,14.70,3,2,49.049049
2405,2431,"Litware 16"" White Oscillating Stand Fan E701 W...",Litware,Home Appliances,29.09,14.26,1,1,49.020282
917,919,SV USB Data Cable E600 Pink,Southridge Video,Computers,25.65,12.69,27,9,49.473684
920,922,SV USB Data Cable E600 Grey,Southridge Video,Computers,21.85,10.81,23,6,49.473684
2419,2445,Litware 80mm Dual Ball Bearing Case Fan E1001 ...,Litware,Home Appliances,19.96,9.80,4,2,49.098196
924,926,SV USB Sync Charge Cable E700 Silver,Southridge Video,Computers,15.92,7.84,8,4,49.246231
919,921,SV USB Data Cable E600 Silver,Southridge Video,Computers,15.20,7.52,16,5,49.473684


In [8]:
store_performance = pd.DataFrame()

if "store_key" in sales.columns:
    store_group_columns = ["store_key"]

    for column in ["store_name", "store_city", "store_state", "store_country"]:
        if column in sales.columns:
            store_group_columns.append(column)

    store_performance = (
        sales.groupby(store_group_columns, dropna=False)
        .agg(
            revenue_usd=("revenue_usd", "sum"),
            gross_profit_usd=("gross_profit_usd", "sum"),
            units_sold=("quantity", "sum"),
            transaction_rows=("quantity", "size")
        )
        .reset_index()
    )

    store_performance["gross_margin_pct"] = np.where(
        store_performance["revenue_usd"] != 0,
        store_performance["gross_profit_usd"]
        / store_performance["revenue_usd"] * 100,
        np.nan
    )

    store_performance["revenue_contribution_pct"] = (
        store_performance["revenue_usd"] / total_revenue * 100
        if total_revenue != 0 else np.nan
    )

    store_performance = store_performance.sort_values(
        "revenue_usd", ascending=False
    )

    print("Top 10 stores by revenue")
    display(store_performance.head(10))

    print("Bottom 10 stores by revenue")
    display(store_performance.tail(10))

else:
    print("Store key unavailable. Store analysis skipped.")

Top 10 stores by revenue


,store_key,revenue_usd,gross_profit_usd,units_sold,transaction_rows,gross_margin_pct,revenue_contribution_pct
0,0,11404324.63,6672692.64,41311,13165,58.510196,20.454177
48,55,1417885.41,830551.75,4826,1518,58.576789,2.543042
44,50,1394738.06,819659.12,4773,1519,58.767961,2.501526
47,54,1384396.24,810300.06,4686,1498,58.530935,2.482978
7,9,1336150.06,774757.55,4894,1577,57.984322,2.396446
50,57,1325611.89,784095.76,4407,1442,59.149723,2.377545
40,45,1318787.92,761271.82,4672,1471,57.725113,2.365306
52,61,1305684.43,764997.20,4771,1485,58.589747,2.341805
51,59,1302272.44,754702.32,4674,1472,57.952721,2.335685
55,64,1259350.98,745283.99,4289,1353,59.180006,2.258704


Bottom 10 stores by revenue


,store_key,revenue_usd,gross_profit_usd,units_sold,transaction_rows,gross_margin_pct,revenue_contribution_pct
1,1,243029.93,141355.64,871,292,58.163881,0.435885
15,18,226078.88,135077.79,607,194,59.748080,0.405483
12,15,205119.67,118097.69,715,235,57.575019,0.367891
13,16,199009.69,119915.80,683,214,60.256262,0.356933
24,28,187109.49,108108.47,660,214,57.778186,0.335589
9,12,183091.04,107084.53,748,224,58.487040,0.328382
14,17,159607.50,92449.65,626,210,57.923124,0.286263
10,13,150925.12,88957.65,592,184,58.941580,0.270691
11,14,105714.05,63591.55,361,126,60.154303,0.189603
2,2,15175.99,9512.61,61,18,62.681973,0.027219


In [9]:
customer_performance = pd.DataFrame()

if "customer_key" in sales.columns:
    customer_aggregations = {
        "revenue_usd": ("revenue_usd", "sum"),
        "units_purchased": ("quantity", "sum"),
        "first_purchase": ("order_date", "min"),
        "last_purchase": ("order_date", "max")
    }

    if "gross_profit_usd" in sales.columns:
        customer_aggregations["gross_profit_usd"] = (
            "gross_profit_usd", "sum"
        )

    if "order_number" in sales.columns:
        customer_aggregations["orders"] = ("order_number", "nunique")
    else:
        customer_aggregations["orders"] = ("order_date", "count")

    customer_performance = (
        sales.groupby("customer_key")
        .agg(**customer_aggregations)
        .reset_index()
    )

    reference_date = sales["order_date"].max() + pd.Timedelta(days=1)

    customer_performance["recency_days"] = (
        reference_date - customer_performance["last_purchase"]
    ).dt.days

    customer_performance["average_order_value"] = (
        customer_performance["revenue_usd"]
        / customer_performance["orders"].replace(0, np.nan)
    )

    customer_performance = customer_performance.sort_values(
        "revenue_usd", ascending=False
    )

    print("Top 10 customers by total revenue")
    display(customer_performance.head(10))

    print("\nCustomer value summary")
    display(
        customer_performance[
            ["revenue_usd", "orders", "average_order_value", "recency_days"]
        ].describe()
    )

else:
    print("Customer key unavailable. Customer analysis skipped.")

Top 10 customers by total revenue


,customer_key,revenue_usd,units_purchased,first_purchase,last_purchase,gross_profit_usd,orders,recency_days,average_order_value
9371,1702221,61871.70,75,2016-05-02,2019-12-27,39606.02,9,422,6874.633333
10515,1884663,43517.80,30,2018-11-12,2019-02-20,28819.96,3,732,14505.933333
11072,1969704,42788.04,60,2016-08-13,2020-01-02,26008.23,7,416,6112.577143
2743,535496,41521.53,35,2016-12-24,2018-11-24,26957.37,3,820,13840.510000
2840,551036,40556.54,36,2017-12-12,2020-12-25,26716.09,4,58,10139.135000
3664,723572,40225.01,124,2016-02-20,2021-02-20,23440.78,14,1,2873.215000
1175,262871,38813.88,32,2017-12-25,2020-02-20,23986.19,3,367,12937.960000
10795,1928466,38191.06,40,2016-08-15,2019-11-21,24054.60,4,458,9547.765000
8116,1503831,37319.88,29,2018-07-18,2018-12-26,22881.47,3,788,12439.960000
7275,1373561,36817.28,62,2017-01-26,2019-11-23,22954.48,5,456,7363.456000



Customer value summary


,revenue_usd,orders,average_order_value,recency_days
count,11887.000000,11887.000000,11887.000000,11887.000000
mean,4690.458450,2.214688,2122.457421,615.142088
std,5197.622706,1.391487,2251.639071,406.202795
min,1.990000,1.000000,1.990000,1.000000
25%,1138.915000,1.000000,713.959000,367.000000
50%,2995.000000,2.000000,1503.280000,509.000000
75%,6425.065000,3.000000,2723.533333,796.000000
max,61871.700000,14.000000,32056.890000,1878.000000


In [10]:
rfm = pd.DataFrame()

if not customer_performance.empty:
    rfm = customer_performance[
        ["customer_key", "recency_days", "orders", "revenue_usd"]
    ].copy()

    rfm = rfm.rename(columns={
        "recency_days": "recency",
        "orders": "frequency",
        "revenue_usd": "monetary"
    })

    # Rank values first to reduce problems caused by duplicate quantile edges
    number_of_customers = len(rfm)

    if number_of_customers >= 5:
        rfm["R_score"] = pd.qcut(
            rfm["recency"].rank(method="first"),
            5,
            labels=[5, 4, 3, 2, 1]
        ).astype(int)

        rfm["F_score"] = pd.qcut(
            rfm["frequency"].rank(method="first"),
            5,
            labels=[1, 2, 3, 4, 5]
        ).astype(int)

        rfm["M_score"] = pd.qcut(
            rfm["monetary"].rank(method="first"),
            5,
            labels=[1, 2, 3, 4, 5]
        ).astype(int)

        def assign_segment(row):
            if row["R_score"] >= 4 and row["F_score"] >= 4 and row["M_score"] >= 4:
                return "Champions"
            elif row["R_score"] >= 3 and row["F_score"] >= 4:
                return "Loyal Customers"
            elif row["R_score"] >= 4 and row["F_score"] <= 2:
                return "Potential Loyalists"
            elif row["R_score"] <= 2 and row["F_score"] >= 3:
                return "At-Risk Customers"
            elif row["R_score"] <= 2 and row["F_score"] <= 2:
                return "Hibernating Customers"
            else:
                return "Other Customers"

        rfm["customer_segment"] = rfm.apply(assign_segment, axis=1)

        segment_performance = (
            rfm.groupby("customer_segment")
            .agg(
                customer_count=("customer_key", "nunique"),
                average_recency=("recency", "mean"),
                average_frequency=("frequency", "mean"),
                total_customer_revenue=("monetary", "sum"),
                average_customer_revenue=("monetary", "mean")
            )
            .reset_index()
        )

        segment_performance["revenue_contribution_pct"] = (
            segment_performance["total_customer_revenue"]
            / total_revenue * 100
            if total_revenue != 0 else np.nan
        )

        segment_performance = segment_performance.sort_values(
            "total_customer_revenue", ascending=False
        )

        display(segment_performance)

    else:
        print("At least five customers are required for five-level RFM scoring.")
        segment_performance = pd.DataFrame()

else:
    segment_performance = pd.DataFrame()
    print("Customer data is unavailable. RFM analysis skipped.")

,customer_segment,customer_count,average_recency,average_frequency,total_customer_revenue,average_customer_revenue,revenue_contribution_pct
1,Champions,1781,260.071870,4.253790,18769822.54,10538.923380,33.664534
0,At-Risk Customers,1860,854.016667,2.480645,9293494.56,4996.502452,16.668307
4,Other Customers,2247,424.129506,1.668002,8896758.68,3959.394161,15.956743
3,Loyal Customers,2017,392.782350,3.119980,8352149.71,4140.877397,14.979962
2,Hibernating Customers,2895,1105.036960,1.018653,7414483.45,2561.134180,13.298215
5,Potential Loyalists,1087,290.885005,1.054278,3028770.65,2786.357544,5.432239


In [11]:
business_findings = []

def add_finding(area, finding, evidence, implication):
    business_findings.append({
        "Business Area": area,
        "Finding": finding,
        "Quantitative Evidence": evidence,
        "Business Implication": implication
    })

# Overall profitability
if pd.notna(overall_margin):
    add_finding(
        "Overall Performance",
        "The business has an overall gross profit margin of "
        f"{overall_margin:.2f}%.",
        f"Revenue: ${total_revenue:,.2f}; Gross profit: ${total_profit:,.2f}.",
        "Management should monitor margin alongside revenue to ensure sales growth is profitable."
    )

# Monthly sales
if not monthly_sales.empty:
    add_finding(
        "Sales Trends",
        f"The highest-revenue month was {best_month['year_month']}.",
        f"Revenue reached ${best_month['revenue_usd']:,.2f}.",
        "Investigate the drivers of peak-period sales and assess whether successful campaigns or seasonal patterns can be repeated."
    )

    if len(monthly_sales) > 1:
        latest_month = monthly_sales.iloc[-1]
        previous_month = monthly_sales.iloc[-2]

        growth = latest_month["revenue_growth_pct"]

        if pd.notna(growth):
            direction = "increased" if growth >= 0 else "decreased"

            add_finding(
                "Recent Sales Trends",
                f"Revenue {direction} by {abs(growth):.2f}% in the latest month compared with the previous month.",
                f"Latest month: {latest_month['year_month']}; "
                f"Revenue: ${latest_month['revenue_usd']:,.2f}.",
                "Review recent sales activity and investigate the factors behind the change."
            )

# Category performance
if not category_performance.empty:
    top_category = category_performance.iloc[0]

    add_finding(
        "Product Categories",
        f"{top_category['category']} generated the highest category revenue.",
        f"Revenue: ${top_category['revenue_usd']:,.2f}; "
        f"Revenue contribution: {top_category['revenue_contribution_pct']:.2f}%.",
        "Protect availability of high-performing products in this category while assessing opportunities for profitable expansion."
    )

    low_margin_categories = category_performance[
        category_performance["gross_margin_pct"] < overall_margin
    ] if pd.notna(overall_margin) else pd.DataFrame()

    if not low_margin_categories.empty:
        names = ", ".join(
            low_margin_categories["category"].astype(str).head(5)
        )

        add_finding(
            "Category Profitability",
            "Some categories generated below-average gross profit margins.",
            f"Categories below the overall margin: {names}.",
            "Investigate supplier costs, pricing, discounts and product mix in these categories."
        )

# Product performance
if not product_performance.empty:
    top_product = product_performance.iloc[0]

    product_label = (
        str(top_product["product_name"])
        if "product_name" in product_performance.columns
        else str(top_product["product_key"])
    )

    add_finding(
        "Product Performance",
        f"{product_label} was the highest-revenue product.",
        f"Revenue: ${top_product['revenue_usd']:,.2f}; "
        f"Units sold: {top_product['units_sold']:,.0f}.",
        "Review stock availability and maintain appropriate inventory levels for high-demand products."
    )

# Store performance
if not store_performance.empty:
    top_store = store_performance.iloc[0]

    add_finding(
        "Store Performance",
        f"Store {top_store['store_key']} generated the highest revenue.",
        f"Revenue: ${top_store['revenue_usd']:,.2f}; "
        f"Contribution: {top_store['revenue_contribution_pct']:.2f}%.",
        "Examine the practices and operating conditions of the leading store to identify approaches that may be transferable."
    )

    if pd.notna(overall_margin):
        below_average_stores = store_performance[
            store_performance["gross_margin_pct"] < overall_margin
        ]

        if not below_average_stores.empty:
            add_finding(
                "Store Profitability",
                f"{len(below_average_stores)} stores had a gross margin below the business average.",
                f"Business gross margin: {overall_margin:.2f}%.",
                "Review pricing, discounting, product mix and operating practices at stores with lower margins."
            )

# Customer performance
if not customer_performance.empty:
    top_customer = customer_performance.iloc[0]

    add_finding(
        "Customer Value",
        f"Customer {top_customer['customer_key']} generated the highest recorded customer revenue.",
        f"Customer revenue: ${top_customer['revenue_usd']:,.2f}.",
        "Use customer value analysis to inform retention and loyalty initiatives, while avoiding over-reliance on a small number of customers."
    )

if not rfm.empty and "customer_segment" in rfm.columns:
    at_risk_count = (
        rfm["customer_segment"] == "At-Risk Customers"
    ).sum()

    add_finding(
        "Customer Retention",
        f"{at_risk_count} customers were classified as at-risk.",
        f"At-risk customers represent "
        f"{at_risk_count / len(rfm) * 100:.2f}% of analysed customers.",
        "Consider targeted re-engagement campaigns and investigate why previously active customers have reduced their purchasing activity."
    )

findings_df = pd.DataFrame(business_findings)

display(findings_df)

,Business Area,Finding,Quantitative Evidence,Business Implication
0,Overall Performance,The business has an overall gross profit margi...,"Revenue: $55,755,479.59; Gross profit: $32,662...",Management should monitor margin alongside rev...
1,Sales Trends,The highest-revenue month was 2019-12.,"Revenue reached $2,477,295.85.",Investigate the drivers of peak-period sales a...
2,Recent Sales Trends,Revenue increased by 2.58% in the latest month...,"Latest month: 2021-02; Revenue: $526,266.90.",Review recent sales activity and investigate t...
3,Product Categories,Computers generated the highest category revenue.,"Revenue: $19,301,595.46; Revenue contribution:...",Protect availability of high-performing produc...
4,Category Profitability,Some categories generated below-average gross ...,Categories below the overall margin: Computers...,"Investigate supplier costs, pricing, discounts..."
5,Product Performance,WWI Desktop PC2.33 X2330 Black was the highest...,"Revenue: $505,450.00; Units sold: 550.",Review stock availability and maintain appropr...
6,Store Performance,Store 0.0 generated the highest revenue.,"Revenue: $11,404,324.63; Contribution: 20.45%.",Examine the practices and operating conditions...
7,Store Profitability,29 stores had a gross margin below the busines...,Business gross margin: 58.58%.,"Review pricing, discounting, product mix and o..."
8,Customer Value,Customer 1702221 generated the highest recorde...,"Customer revenue: $61,871.70.",Use customer value analysis to inform retentio...
9,Customer Retention,1860 customers were classified as at-risk.,At-risk customers represent 15.65% of analysed...,Consider targeted re-engagement campaigns and ...


In [12]:
recommendations = []

def add_recommendation(priority, area, recommendation, evidence, expected_benefit):
    recommendations.append({
        "Priority": priority,
        "Business Area": area,
        "Recommendation": recommendation,
        "Supporting Evidence": evidence,
        "Expected Business Benefit": expected_benefit
    })

# Overall profitability
if pd.notna(overall_margin):
    add_recommendation(
        "High",
        "Profitability",
        "Introduce regular gross margin monitoring across products, categories and stores.",
        f"Current overall gross profit margin: {overall_margin:.2f}%.",
        "Improved visibility of profitability and earlier identification of margin deterioration."
    )

# Low-margin categories
if not category_performance.empty and pd.notna(overall_margin):
    low_margin_categories = category_performance[
        category_performance["gross_margin_pct"] < overall_margin
    ]

    if not low_margin_categories.empty:
        category_names = ", ".join(
            low_margin_categories["category"].astype(str).head(5)
        )

        add_recommendation(
            "High",
            "Category Profitability",
            "Review supplier pricing, selling prices, promotional discounts and product mix in below-average-margin categories.",
            f"Categories with margins below {overall_margin:.2f}%: {category_names}.",
            "Potential improvement in gross margin and more effective pricing decisions."
        )

# High-performing products
if not product_performance.empty:
    top_products = product_performance.head(10)

    add_recommendation(
        "High",
        "Product Management",
        "Prioritise availability and monitor inventory levels for the highest-revenue products.",
        f"The top product generated ${top_products.iloc[0]['revenue_usd']:,.2f} in revenue.",
        "Reduced risk of stockouts and improved ability to meet customer demand."
    )

    if pd.notna(overall_margin):
        low_performing_products = product_performance[
            (product_performance["revenue_usd"] < product_performance["revenue_usd"].median()) &
            (product_performance["gross_margin_pct"] < overall_margin)
        ]

        if not low_performing_products.empty:
            add_recommendation(
                "Medium",
                "Product Portfolio",
                "Review products with both below-median revenue and below-average gross margins.",
                f"{len(low_performing_products)} products meet both conditions.",
                "Better product assortment decisions and reduced attention on weak commercial performers."
            )

# Store performance
if not store_performance.empty and pd.notna(overall_margin):
    weak_stores = store_performance[
        store_performance["gross_margin_pct"] < overall_margin
    ]

    if not weak_stores.empty:
        add_recommendation(
            "High",
            "Store Management",
            "Conduct a performance review of stores with gross margins below the business average.",
            f"{len(weak_stores)} stores are below the overall margin of {overall_margin:.2f}%.",
            "Identification of store-level pricing, product mix or operational issues."
        )

    add_recommendation(
        "Medium",
        "Store Benchmarking",
        "Compare the sales and profitability practices of high-performing stores with lower-performing stores.",
        f"Highest-revenue store: {store_performance.iloc[0]['store_key']}.",
        "Transfer of effective practices and more consistent store performance."
    )

# Customer retention
if not rfm.empty and "customer_segment" in rfm.columns:
    at_risk = rfm[rfm["customer_segment"] == "At-Risk Customers"]

    if not at_risk.empty:
        add_recommendation(
            "High",
            "Customer Retention",
            "Develop targeted re-engagement campaigns for at-risk customers.",
            f"{len(at_risk)} customers were classified as at-risk.",
            "Potential recovery of customer activity and protection of existing revenue."
        )

    champions = rfm[rfm["customer_segment"] == "Champions"]

    if not champions.empty:
        add_recommendation(
            "Medium",
            "Customer Loyalty",
            "Strengthen loyalty initiatives for champion customers through personalised offers and retention programmes.",
            f"{len(champions)} customers were classified as champions.",
            "Improved customer retention and opportunities to increase customer lifetime value."
        )

# Sales trends
if len(monthly_sales) > 1:
    latest_growth = monthly_sales.iloc[-1]["revenue_growth_pct"]

    if pd.notna(latest_growth) and latest_growth < 0:
        add_recommendation(
            "High",
            "Sales Growth",
            "Investigate the latest month-on-month revenue decline by examining product, store and customer-level changes.",
            f"Latest monthly revenue change: {latest_growth:.2f}%.",
            "Earlier identification of declining sales drivers and more informed corrective action."
        )

    add_recommendation(
        "Medium",
        "Sales Planning",
        "Use monthly sales patterns to support promotional planning and demand forecasting.",
        f"The dataset contains {len(monthly_sales)} monthly periods.",
        "Improved planning around periods of stronger and weaker sales activity."
    )

recommendations_df = pd.DataFrame(recommendations)

priority_order = {"High": 1, "Medium": 2, "Low": 3}

if not recommendations_df.empty:
    recommendations_df["_priority_order"] = (
        recommendations_df["Priority"].map(priority_order)
    )

    recommendations_df = (
        recommendations_df
        .sort_values("_priority_order")
        .drop(columns="_priority_order")
        .reset_index(drop=True)
    )

display(recommendations_df)

,Priority,Business Area,Recommendation,Supporting Evidence,Expected Business Benefit
0,High,Profitability,Introduce regular gross margin monitoring acro...,Current overall gross profit margin: 58.58%.,Improved visibility of profitability and earli...
1,High,Category Profitability,"Review supplier pricing, selling prices, promo...",Categories with margins below 58.58%: Computer...,Potential improvement in gross margin and more...
2,High,Product Management,Prioritise availability and monitor inventory ...,"The top product generated $505,450.00 in revenue.",Reduced risk of stockouts and improved ability...
3,High,Store Management,Conduct a performance review of stores with gr...,29 stores are below the overall margin of 58.58%.,"Identification of store-level pricing, product..."
4,High,Customer Retention,Develop targeted re-engagement campaigns for a...,1860 customers were classified as at-risk.,Potential recovery of customer activity and pr...
5,Medium,Product Portfolio,Review products with both below-median revenue...,1152 products meet both conditions.,Better product assortment decisions and reduce...
6,Medium,Store Benchmarking,Compare the sales and profitability practices ...,Highest-revenue store: 0.0.,Transfer of effective practices and more consi...
7,Medium,Customer Loyalty,Strengthen loyalty initiatives for champion cu...,1781 customers were classified as champions.,Improved customer retention and opportunities ...
8,Medium,Sales Planning,Use monthly sales patterns to support promotio...,The dataset contains 62 monthly periods.,Improved planning around periods of stronger a...


In [13]:
decision_matrix = pd.DataFrame([
    {
        "Business Challenge": "Declining monthly sales",
        "Management Action": "Investigate sales changes by product, store and customer.",
        "Suggested KPI": "Month-on-month revenue growth",
        "Review Frequency": "Monthly"
    },
    {
        "Business Challenge": "Low product profitability",
        "Management Action": "Review pricing, supplier costs and discounts.",
        "Suggested KPI": "Gross profit margin",
        "Review Frequency": "Monthly"
    },
    {
        "Business Challenge": "Uneven store performance",
        "Management Action": "Benchmark stores and investigate performance gaps.",
        "Suggested KPI": "Store revenue and gross margin",
        "Review Frequency": "Monthly"
    },
    {
        "Business Challenge": "Customer inactivity",
        "Management Action": "Target at-risk customers with retention campaigns.",
        "Suggested KPI": "Customer retention and repeat purchase rate",
        "Review Frequency": "Monthly or quarterly"
    },
    {
        "Business Challenge": "High demand for selected products",
        "Management Action": "Monitor stock availability and plan replenishment.",
        "Suggested KPI": "Units sold and product stockout rate",
        "Review Frequency": "Weekly"
    },
    {
        "Business Challenge": "Unclear commercial priorities",
        "Management Action": "Use a management scorecard to monitor key business measures.",
        "Suggested KPI": "Revenue, gross profit and gross margin",
        "Review Frequency": "Monthly"
    }
])

display(decision_matrix)

,Business Challenge,Management Action,Suggested KPI,Review Frequency
0,Declining monthly sales,"Investigate sales changes by product, store an...",Month-on-month revenue growth,Monthly
1,Low product profitability,"Review pricing, supplier costs and discounts.",Gross profit margin,Monthly
2,Uneven store performance,Benchmark stores and investigate performance g...,Store revenue and gross margin,Monthly
3,Customer inactivity,Target at-risk customers with retention campai...,Customer retention and repeat purchase rate,Monthly or quarterly
4,High demand for selected products,Monitor stock availability and plan replenishm...,Units sold and product stockout rate,Weekly
5,Unclear commercial priorities,Use a management scorecard to monitor key busi...,"Revenue, gross profit and gross margin",Monthly


In [14]:
limitations = pd.DataFrame({
    "Limitation": [
        "Historical data",
        "Gross profit measure",
        "Inventory availability",
        "Customer segmentation",
        "Causal relationships",
        "External business factors"
    ],
    "Explanation": [
        "Historical performance does not guarantee future results.",
        "Gross profit is based on available product cost and selling price data. It may not include rent, salaries, distribution or other operating expenses.",
        "The dataset does not necessarily contain stock-on-hand or stockout records, so inventory recommendations should be validated against operational inventory data.",
        "RFM segments describe purchasing behaviour in the available dataset and should not be treated as permanent customer characteristics.",
        "The analysis identifies patterns and associations but does not establish that one factor caused another.",
        "Competitor activity, inflation, exchange-rate movements and broader market conditions may influence performance but may not be represented in the dataset."
    ]
})

display(limitations)

,Limitation,Explanation
0,Historical data,Historical performance does not guarantee futu...
1,Gross profit measure,Gross profit is based on available product cos...
2,Inventory availability,The dataset does not necessarily contain stock...
3,Customer segmentation,RFM segments describe purchasing behaviour in ...
4,Causal relationships,The analysis identifies patterns and associati...
5,External business factors,"Competitor activity, inflation, exchange-rate ..."


In [15]:
# Export analytical tables
exports = {
    "business_baseline.csv": business_baseline,
    "monthly_sales_analysis.csv": monthly_sales,
    "category_performance.csv": category_performance,
    "product_performance.csv": product_performance,
    "store_performance.csv": store_performance,
    "customer_performance.csv": customer_performance,
    "customer_rfm_analysis.csv": rfm,
    "business_findings.csv": findings_df,
    "business_recommendations.csv": recommendations_df,
    "management_decision_matrix.csv": decision_matrix,
    "analytical_limitations.csv": limitations
}

for filename, dataframe in exports.items():
    dataframe.to_csv(OUTPUT_DIR / filename, index=False)
    print(f"Exported: {filename}")

print("\nAll business insight outputs exported successfully.")
print("Output folder:", OUTPUT_DIR)

Exported: business_baseline.csv
Exported: monthly_sales_analysis.csv
Exported: category_performance.csv
Exported: product_performance.csv
Exported: store_performance.csv
Exported: customer_performance.csv
Exported: customer_rfm_analysis.csv
Exported: business_findings.csv
Exported: business_recommendations.csv
Exported: management_decision_matrix.csv
Exported: analytical_limitations.csv

All business insight outputs exported successfully.
Output folder: C:\Users\Administrator\retail-sales-intelligence\data\processed\business_insights_outputs


In [16]:
summary_lines = []

summary_lines.append("# Retail Sales Intelligence — Executive Summary\n")

summary_lines.append("## Overall Business Performance\n")

summary_lines.append(
    f"- Total revenue: ${total_revenue:,.2f}"
)

summary_lines.append(
    f"- Total gross profit: ${total_profit:,.2f}"
    if pd.notna(total_profit)
    else "- Total gross profit: Not available"
)

summary_lines.append(
    f"- Gross profit margin: {overall_margin:.2f}%"
    if pd.notna(overall_margin)
    else "- Gross profit margin: Not available"
)

summary_lines.append(f"- Units sold: {total_units:,.0f}")

summary_lines.append("\n## Key Business Findings\n")

if not findings_df.empty:
    for index, row in findings_df.iterrows():
        summary_lines.append(
            f"{index + 1}. **{row['Business Area']}** — "
            f"{row['Finding']} "
            f"Evidence: {row['Quantitative Evidence']}"
        )
else:
    summary_lines.append("No findings were generated.")

summary_lines.append("\n## Priority Recommendations\n")

if not recommendations_df.empty:
    for index, row in recommendations_df.iterrows():
        summary_lines.append(
            f"{index + 1}. **{row['Priority']} priority — "
            f"{row['Business Area']}:** "
            f"{row['Recommendation']} "
            f"Supporting evidence: {row['Supporting Evidence']}"
        )
else:
    summary_lines.append("No recommendations were generated.")

summary_lines.append("\n## Important Considerations\n")

for _, row in limitations.iterrows():
    summary_lines.append(
        f"- **{row['Limitation']}:** {row['Explanation']}"
    )

summary_text = "\n\n".join(summary_lines)

summary_file = OUTPUT_DIR / "executive_business_summary.md"
summary_file.write_text(summary_text, encoding="utf-8")

print("Executive summary created:")
print(summary_file)

print("\n" + summary_text)

Executive summary created:
C:\Users\Administrator\retail-sales-intelligence\data\processed\business_insights_outputs\executive_business_summary.md

# Retail Sales Intelligence — Executive Summary


## Overall Business Performance


- Total revenue: $55,755,479.59

- Total gross profit: $32,662,688.38

- Gross profit margin: 58.58%

- Units sold: 197,757


## Key Business Findings


1. **Overall Performance** — The business has an overall gross profit margin of 58.58%. Evidence: Revenue: $55,755,479.59; Gross profit: $32,662,688.38.

2. **Sales Trends** — The highest-revenue month was 2019-12. Evidence: Revenue reached $2,477,295.85.

3. **Recent Sales Trends** — Revenue increased by 2.58% in the latest month compared with the previous month. Evidence: Latest month: 2021-02; Revenue: $526,266.90.

4. **Product Categories** — Computers generated the highest category revenue. Evidence: Revenue: $19,301,595.46; Revenue contribution: 34.62%.

5. **Category Profitability** — Some categories 

## Conclusion

This notebook translated the retail sales analysis into practical business insights and management recommendations.

The analysis examined revenue generation, gross profitability, monthly sales trends, product and category performance, store performance and customer purchasing behaviour.

The findings provide a foundation for management to:
- Improve profitability through better pricing and product mix decisions.
- Strengthen inventory planning for high-demand products.
- Identify and investigate underperforming stores and categories.
- Improve customer retention through behavioural segmentation.
- Monitor commercial performance using measurable KPIs.

The recommendations should be reviewed alongside operational information such as inventory records, operating expenses, marketing expenditure and supplier costs before implementation.

### Project outcome

The Retail Sales Intelligence project demonstrates how data preparation, exploratory analysis, commercial performance measurement and business intelligence can be combined to support evidence-based retail decision-making.